### FASE 3: División de Datos 

| Vamos a extraer caracteristicas por cada registro

Importamos librerias

In [85]:
import pandas as pd
import numpy as np
from tqdm.notebook import tqdm
tqdm.pandas()
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import TimeSeriesSplit,cross_val_score
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import pyswarms as ps
from pyswarms.utils.functions import single_obj as fx
import seaborn as sns
import matplotlib.pyplot as plt

In [86]:
df = pd.read_csv('completo_clusters_con_features.csv')

df_model = df.copy()

le = LabelEncoder()
df_model['Direccion'] = le.fit_transform(df_model['Direccion'])


features = [
    'Total_Vehiculos','Tiempo_Medio_s', 'Ocupacion_Espacial_%', 
    'Hora_Minutos', 'Dia_Semana', 'Direccion',
    'Es_Hora_Pico', 'Total_Vehiculos_lag1', 'Ocupacion_lag1', 
    'Media_Movil_3ciclos', 'Tendencia_Vehiculos', 'Saturacion_Actual',
    'Periodo_Dia', 'Cluster_Hora_Pico' 
]


entreno_data = df_model[df_model['Dia_Semana'].isin([1, 2,5])]
val_data   = df_model[df_model['Dia_Semana'] == 3]
prueba_data  = df_model[df_model['Dia_Semana'] == 4]


X_entreno, y_entreno = entreno_data[features], entreno_data['Tiempo_Optimo_Target']
X_val, y_val     = val_data[features], val_data['Tiempo_Optimo_Target']
X_prueba, y_prueba   = prueba_data[features], prueba_data['Tiempo_Optimo_Target']

df_model = df_model.sort_values(by=['Dia_Semana', 'Hora_Minutos']).reset_index(drop=True)

X = df_model[features]
y = df_model['Tiempo_Optimo_Target']


df_model  = pd.concat([entreno_data, val_data, prueba_data])


### FASE 4: Optimización con PSO

In [87]:

def pso_rf_optimization_robust(X, y, n_particles=12, n_iters=1):
    """
    Usa PSO para optimizar hiperparámetros de Random Forest usando Cross-Validation.
    
    Args:
        X: Dataset completo de características (Training + Validation)
        y: Dataset completo de target
    """
    
    # Definimos las opciones de max_features fuera para no recrearlas
    # 1.0 equivale a 'None' (usar todos los features) en versiones nuevas de sklearn
    MAX_FEAT_OPTS = ['sqrt', 'log2', 1.0] 

    def fitness_function(particles):
        """
        Función de costo: Retorna el MSE promedio de Cross-Validation (negativo invertido)
        """
        scores = []
        
        for particle in particles:
            # 1. Decodificar los parámetros de la partícula (float -> int/cat)
            n_estimators = int(particle[0])
            max_depth = int(particle[1])
            min_samples_split = int(particle[2])
            min_samples_leaf = int(particle[3])
            
            # Mapeo seguro del índice para max_features
            feat_idx = int(particle[4])
            if feat_idx >= len(MAX_FEAT_OPTS): feat_idx = len(MAX_FEAT_OPTS) - 1
            max_features = MAX_FEAT_OPTS[feat_idx]
            
            # Si max_depth es muy bajo (< 2), lo tratamos como None (profundidad ilimitada)
            final_depth = max_depth if max_depth > 1 else None

            # 2. Configurar el modelo
            rf = RandomForestRegressor(
                n_estimators=n_estimators,
                max_depth=final_depth,
                min_samples_split=min_samples_split,
                min_samples_leaf=min_samples_leaf,
                max_features=max_features,
                random_state=42,
                n_jobs=-1 # Usa todos los núcleos para el entrenamiento
            )
            
            # 3. Validación Cruzada (K-Fold)
            # Esto es lo que "tarda más" pero asegura calidad.
            # cv=5 divide los datos en 5 partes y prueba en todas.
            try:
                # cross_val_score devuelve 'neg_mean_squared_error', por eso multiplicamos por -1
                cv_scores = cross_val_score(rf, X, y, cv=5, scoring='neg_mean_squared_error', n_jobs=-1)
                mean_mse = -1 * np.mean(cv_scores)
                scores.append(mean_mse)
            except Exception as e:
                # Penalización suave en lugar de gigantesca para no romper la inercia del enjambre
                scores.append(1e5) 
        
        return np.array(scores)
    
    # --- Definición de Límites (Bounds) ---
    # [n_estimators, max_depth, min_samples_split, min_samples_leaf, max_features_idx]
    lb = [100,  2, 2,  1, 0]    # Mínimos (Más árboles = más estabilidad)
    ub = [1000, 50, 20, 10, 2.99] # Máximos (Índice features llega casi a 3)
    bounds = (lb, ub)
    
    # --- Configuración del PSO ---
    # c1 (cognitivo): cuánto confía en su propia mejor posición
    # c2 (social): cuánto confía en el mejor del enjambre
    # w (inercia): cuánto mantiene su velocidad actual
    options = {'c1': 0.6, 'c2': 0.4, 'w': 0.8} 
    
    optimizer = ps.single.GlobalBestPSO(
        n_particles=n_particles, 
        dimensions=5,
        options=options,
        bounds=bounds
    )
    
    print(f"Iniciando optimización robusta con {n_particles} partículas y {n_iters} iteraciones...")
    print("Nota: Esto tardará porque usa 5-Fold Cross Validation.")
    
    best_cost, best_pos = optimizer.optimize(fitness_function, iters=n_iters)
    
    # Decodificar la mejor posición encontrada
    final_feat_idx = int(best_pos[4])
    if final_feat_idx >= len(MAX_FEAT_OPTS): final_feat_idx = len(MAX_FEAT_OPTS) - 1
    
    best_params = {
        'n_estimators': int(best_pos[0]),
        'max_depth': int(best_pos[1]) if int(best_pos[1]) > 1 else None,
        'min_samples_split': int(best_pos[2]),
        'min_samples_leaf': int(best_pos[3]),
        'max_features': MAX_FEAT_OPTS[final_feat_idx]
    }
    
    return best_params, best_cost

best_params, best_cost = pso_rf_optimization_robust(X, y)

print("\n=== MEJORES HIPERPARÁMETROS (Validación Cruzada) ===")
print(best_params)
print(f"Mejor MSE promedio (CV): {best_cost:.4f}")
print(f"RMSE estimado: {np.sqrt(best_cost):.4f}")

2026-01-15 16:13:44,229 - pyswarms.single.global_best - INFO - Optimize for 1 iters with {'c1': 0.6, 'c2': 0.4, 'w': 0.8}


Iniciando optimización robusta con 12 partículas y 1 iteraciones...
Nota: Esto tardará porque usa 5-Fold Cross Validation.


pyswarms.single.global_best:   0%|          |0/1

pyswarms.single.global_best: 100%|██████████|1/1, best_cost=6.24
2026-01-15 16:15:29,420 - pyswarms.single.global_best - INFO - Optimization finished | best cost: 6.241248913714889, best pos: [972.85406232  33.58973775  15.46795316   3.34497416   2.41327955]



=== MEJORES HIPERPARÁMETROS (Validación Cruzada) ===
{'n_estimators': 972, 'max_depth': 33, 'min_samples_split': 15, 'min_samples_leaf': 3, 'max_features': 1.0}
Mejor MSE promedio (CV): 6.2412
RMSE estimado: 2.4982


### FASE 5: Modelo Random Forest Optimizado

In [ ]:

tscv = TimeSeriesSplit(n_splits=5)

print(f"Iniciando Validación Cruzada con Expansión (Total muestras: {len(X)})")

mae_scores = []
rmse_scores = []
r2_scores = []

for i, (train_index, test_index) in enumerate(tscv.split(X)):
    X_train_fold, X_test_fold = X.iloc[train_index], X.iloc[test_index]
    y_train_fold, y_test_fold = y.iloc[train_index], y.iloc[test_index]


    rf_fold = RandomForestRegressor(
        **best_params, 
        random_state=42,
        n_jobs=-1
    )
    rf_fold.fit(X_train_fold, y_train_fold)
    
    y_pred_fold = rf_fold.predict(X_test_fold)
    
    mae = mean_absolute_error(y_test_fold, y_pred_fold)
    rmse = np.sqrt(mean_squared_error(y_test_fold, y_pred_fold))
    r2 = r2_score(y_test_fold, y_pred_fold)
    
    mae_scores.append(mae)
    rmse_scores.append(rmse)
    r2_scores.append(r2)
    
    print(f"Iteración {i+1}: Train tam={len(train_index)} | Test tam={len(test_index)} -> MAE: {mae:.3f} | R²: {r2:.4f}")



Iniciando Validación Cruzada con Expansión (Total muestras: 3801)
Iteración 1: Train tam=636 | Test tam=633 -> MAE: 0.793 | R²: 0.9512
Iteración 2: Train tam=1269 | Test tam=633 -> MAE: 0.942 | R²: 0.8802
Iteración 3: Train tam=1902 | Test tam=633 -> MAE: 1.252 | R²: 0.8251
Iteración 4: Train tam=2535 | Test tam=633 -> MAE: 1.047 | R²: 0.8809
Iteración 5: Train tam=3168 | Test tam=633 -> MAE: 1.142 | R²: 0.8756


### FASE 6: Evaluación Final 


In [89]:
print("\n=== RESULTADOS PROMEDIO (VALIDACIÓN ROBUSTA) ===")
print(f"MAE Promedio:  {np.mean(mae_scores):.3f}")
print(f"RMSE Promedio: {np.mean(rmse_scores):.3f}")
print(f"R² Promedio:   {np.mean(r2_scores):.4f}")

rf_final = RandomForestRegressor(**best_params, random_state=42, n_jobs=-1)
rf_final.fit(X, y)
print("\n✅ Modelo final entrenado con el 100% de la historia disponible.")



=== RESULTADOS PROMEDIO (VALIDACIÓN ROBUSTA) ===
MAE Promedio:  1.035
RMSE Promedio: 2.501
R² Promedio:   0.8826

✅ Modelo final entrenado con el 100% de la historia disponible.


### FASE 7: Prediccion de tiempos verdes

In [ ]:

def comparar_vs_tiempo_fijo(test_data, y_test_pred, fijo_verde=30, amarillo=3):
    """
    Compara el modelo dinámico (predicción) vs un ciclo fijo.
    fijo_verde: 30s (El tiempo que realmente fluyen los autos)
    amarillo: 3s (Tiempo muerto/seguridad)
    """
    df = test_data.copy()
    df['Verde_Modelo'] = y_test_pred
    df['Verde_Fijo'] = fijo_verde
    
    df['Diferencia'] = df['Verde_Modelo'] - df['Verde_Fijo']
    

    
    resultados = {}
    
    for direccion in sorted(df['Direccion'].unique()):
        datos = df[df['Direccion'] == direccion]
    
        promedio_modelo = datos['Verde_Modelo'].mean()
        resultados[f'Dir_{direccion}'] = round(promedio_modelo)
        
        print(f"DIRECCIÓN {direccion + 1}")
        print(f"Preccion: {promedio_modelo:.1f}s")

    return resultados

resultados_comparativa = comparar_vs_tiempo_fijo(prueba_data, y_prueba, fijo_verde=30, amarillo=3)



COMPARATIVA: MODELO DINÁMICO vs FIJO (30s verde + 3s amarillo)

🚦 DIRECCIÓN 1
   • Promedio Modelo: 25.9s (vs 30s fijo)
   • 📉 Ahorro (Eficiencia): En el 64.7% de ciclos, ahorramos ~9.6s cada uno.
   • 📈 Congestión (Eficacia): En el 33.2% de ciclos, evitamos colas añadiendo ~6.4s.
----------------------------------------
🚦 DIRECCIÓN 2
   • Promedio Modelo: 27.1s (vs 30s fijo)
   • 📉 Ahorro (Eficiencia): En el 54.7% de ciclos, ahorramos ~9.5s cada uno.
   • 📈 Congestión (Eficacia): En el 44.2% de ciclos, evitamos colas añadiendo ~5.3s.
----------------------------------------
🚦 DIRECCIÓN 3
   • Promedio Modelo: 29.6s (vs 30s fijo)
   • 📉 Ahorro (Eficiencia): En el 39.3% de ciclos, ahorramos ~8.8s cada uno.
   • 📈 Congestión (Eficacia): En el 60.7% de ciclos, evitamos colas añadiendo ~5.0s.
----------------------------------------
🚦 DIRECCIÓN 4
   • Promedio Modelo: 28.5s (vs 30s fijo)
   • 📉 Ahorro (Eficiencia): En el 45.3% de ciclos, ahorramos ~9.3s cada uno.
   • 📈 Congestión (Eficac